# 5 — The word co-occurrence network

The same two-regime structure seen in the word frequencies appears in the
network of word adjacencies, and that is what makes the kernel/periphery reading
structural rather than a property of one statistic.

The network is undirected: nodes are word types, links join tokens adjacent in
the text, weights count co-occurrences. Self-loops are excluded.

Produces **Figure 2**, **Figure S1** and **Table S6**.

In [ ]:
import os
import subprocess
import sys

REPO = os.path.abspath("..") if os.path.isdir(os.path.join("..", "src")) else os.path.abspath(".")
sys.path.insert(0, os.path.join(REPO, "src"))

import plotting as P


def run(*command, must_succeed=True):
    """Run one pipeline step and, unlike a `!` cell, STOP if it fails.

    An IPython `!` cell throws away the exit status: a step that dies leaves no
    output, no error and no trace, and `nbconvert --execute` still reports the
    notebook as successful. Two defects in this pipeline's history hid exactly
    there, so every step below goes through this instead.

    `must_succeed=False` is used only for the two `--check` diagnostics of
    notebook 1, which print a loud banner rather than stopping the run.
    """
    print(">>", " ".join(str(c) for c in command), flush=True)
    code = subprocess.run([str(c) for c in command], check=False).returncode
    if code and must_succeed:
        raise RuntimeError(f"step failed with exit code {code} - read the output "
                           f"above; nothing after this point is valid")
    if code:
        rule = "*" * 72
        print(rule)
        print(f"*** THIS CHECK FAILED (exit code {code}). Read the output above")
        print("*** before going on: whatever depends on this corpus is missing")
        print("*** or wrong, and so is anything computed from it.")
        print(rule, flush=True)
    return code


def py(script, *args, must_succeed=True):
    """`run` for one of this repository's own scripts."""
    return run(sys.executable, os.path.join(REPO, "src", script), *args,
               must_succeed=must_succeed)


%matplotlib inline
USETEX = P.setup_style()
print("repo:", REPO, "| LaTeX text rendering:", USETEX)

## 5.1 Figure 2 — English

Three views of one corpus: the log-binned densities p(k) and p(s), the rank-size
curves k(R) and s(R), and the rank-frequency curves of 1-grams against 2-grams.

The module first checks an identity that ties this figure to Figure 1: a node's
strength counts every bigram position it takes part in, so **s = 2 f** — a word
is once the left member of a bigram and once the right.

Exactly, except at the two ends of a book, where a token has one neighbour
instead of two, and for the self-loops excluded above. That is a minority of
*types* — 5.2% of the English ranks, where a hapax at a book boundary carries
s = 1 rather than 2 — so `check_identity` asserts the **median** of s/2f and
prints its 5th–95th percentiles rather than claiming the identity pointwise.
What does hold is the statement the figure needs: the strength curve of panel B
*is* the Zipf curve of Figure 1A on the same corpus, and the fitted exponents
agree to the two decimals reported — which the module also asserts.

In [ ]:
import figure_2

wcn, gram, degree, strength, freq = figure_2.load()
figure_2.check_identity(wcn, gram, strength, freq)

In [ ]:
table2, pdfs = figure_2.exponents(degree, strength, freq)
figure_2.write_table(table2)
table2

In [ ]:
fig = figure_2.draw(degree, strength, freq, pdfs)
for path in P.save_figure(fig, "fig2"):
    print("wrote", os.path.relpath(path, REPO))

## 5.2 Figure S1 and Table S6 — all five languages

The same three views, each panel holding all five corpora. Panel C is normalised
to relative frequency: the corpora differ in size by a factor of five, and raw
counts would separate the curves by corpus size rather than by shape.

The five curves lying almost on top of one another *is* the result, which is why
this figure draws them thinner than any other.

The table asserts, language by language, that the s(R) columns reproduce the
Table S2 exponents — the same s ≈ 2f identity, now as a five-language regression
check.

In [ ]:
import figure_SI1

wcn5, pdf_k, pdf_s, ngram2 = figure_SI1.load()
figure_SI1.summary(wcn5, ngram2)

In [ ]:
tableS6, bands = figure_SI1.exponents(wcn5, pdf_k, pdf_s, ngram2)
figure_SI1.write_table(tableS6, bands)
figure_SI1.formatted(tableS6, bands)

In [ ]:
fig = figure_SI1.draw(pdf_k, pdf_s, ngram2)
for path in P.save_figure(fig, "SI1"):
    print("wrote", os.path.relpath(path, REPO))

**Next:** `06_phrases_and_concepts.ipynb`.